In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import os

In [2]:
# Set Directory
image_path = "./Dataset"
images = []

for filename in os.listdir(image_path):
    img = tf.keras.preprocessing.image.load_img(f"{image_path}/{filename}", target_size=(102,136))
    img = tf.keras.preprocessing.image.img_to_array(img)
    images.append(img)

In [3]:
# Pre-processing
images = np.array(images).astype(np.float32) / 255.0

x_train = images[:-10] # semua kecuali 10 terakhir
y_train = images[-10:] # 10 images terakhir

In [4]:
# Set constants
BATCH_SIZE = 100
IMAGE_ORIGINAL_SHAPE = x_train.shape[1:] # (102, 136, 3)
IMAGE_SIZE = (
    IMAGE_ORIGINAL_SHAPE[0] * IMAGE_ORIGINAL_SHAPE[1] * IMAGE_ORIGINAL_SHAPE[2]
)
HIDDEN_DIM = 512
LATENT_DIM = 32
MAX_EPOCHS = 10 # Buat di kelas aja, buat hasil bagus ~100
LEARNING_RATE = 0.0001

print(IMAGE_ORIGINAL_SHAPE, IMAGE_SIZE)
# IMAGE_SIZE -> Flatten dari IMAGE_ORIGINAL_SHAPE

(102, 136, 3) 41616


## Model

In [5]:
# # Variational Autoencoder (VAE) Model
# class VAE(tf.keras.Model):
#     def __init__(self, dim, **kwargs):
#         super(VAE, self).__init__(**kwargs)
#         h_dim = dim[0]  # Hidden Dimension
#         z_dim = dim[1]  # Latent Dimension

#         self.fc1 = tf.keras.layers.Dense(h_dim, activation=tf.nn.relu)  # Encoder features
#         self.fc2 = tf.keras.layers.Dense(z_dim)                         # Mean
#         self.fc3 = tf.keras.layers.Dense(z_dim)                         # Log variance
#         self.fc4 = tf.keras.layers.Dense(h_dim, activation=tf.nn.relu)  # Decoder features
#         self.fc5 = tf.keras.layers.Dense(IMAGE_SIZE)                    # Output (logits)

#         # batch normalization
#         self.encoder_batch_norm = tf.keras.layers.BatchNormalization()
#         self.decoder_batch_norm = tf.keras.layers.BatchNormalization()

#     def encoder(self, x, training=None):
#         x = self.encoder_batch_norm(x, training=training)
#         h = self.fc1(x)
#         return self.fc2(h), self.fc3(h)  # mean, log variance

#     def reparameterize(self, mu, log_var):
#         std = tf.exp(log_var * 0.5)                  # std = exp(0.5 * log_var)
#         eps = tf.random.normal(shape=tf.shape(std))  # random noise
#         return mu + eps * std                        # reparameterization trick

#     def decode_logits(self, z, training=None):
#         h = self.fc4(z)
#         h = self.decoder_batch_norm(h, training=training)
#         return self.fc5(h)                           # logits

#     def decode(self, z):
#         return tf.nn.sigmoid(self.decode_logits(z, training=False))  # probabilitas

#     def call(self, inputs, training=None, mask=None):
#         mu, log_var = self.encoder(inputs, training=training)
#         z = self.reparameterize(mu, log_var)
#         x_recon_logits = self.decode_logits(z, training=training)
#         return x_recon_logits, mu, log_var

In [6]:
# ini di lab
# Variational Autoencoder (VAE) Model
class VAE(tf.keras.Model):
    def __init__(self, dim, **kwargs):
        super(VAE, self).__init__(**kwargs)
        h_dim = dim[0] # Hidden Dimension
        z_dim = dim[1] # Latent Dimension

        self.fc1 = tf.keras.layers.Dense(h_dim, activation= tf.nn.relu) # Encoder Features
        self.fc2 = tf.keras.layers.Dense(z_dim) # Cari Mean
        self.fc3 = tf.keras.layers.Dense(z_dim) # Cari Log Variance = standard deviation
        self.fc4 = tf.keras.layers.Dense(h_dim) # Decoder Features
        self.fc5 = tf.keras.layers.Dense(IMAGE_SIZE) # Output
        
        # batch normalization
        self.encoder_batch_norm = tf.keras.layers.BatchNormalization()
        self.decoder_batch_norm = tf.keras.layers.BatchNormalization()

    def encoder(self, x):
        x = self.encoder_batch_norm(x)
        h = self.fc1(x)

        return self.fc2(h), self.fc3(h) # return mean, log variance

    def reparameterize(self, mu, log_var):
        std = tf.exp(log_var * 0.5) # equation for standard deviation
        eps = tf.random.normal(shape=std.shape) # random noise

        return mu + eps * std # equation for reparameterization trick

    def decode_logits(self, z):
        h = self.fc4(z)
        h = self.fc5(h)

        return self.decoder_batch_norm(h) # return logits

    def decode(self, z):
        return tf.nn.sigmoid(self.decode_logits(z)) # return probability

    def call(self, inputs, training=None, mask=None):
        mu, log_var = self.encoder(inputs)
        z = self.reparameterize(mu, log_var)
        x_recon_logits = self.decode(z)

        return x_recon_logits, mu, log_var

In [7]:
# Buat Model
model = VAE((HIDDEN_DIM, LATENT_DIM))
model.build(input_shape=(None, IMAGE_SIZE))  # input: (batch, 41616) = 102*136*3 yang di-flatten
model.summary()

optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)  # untuk update weights

c:\SEM 5\LAB AI Semester 5\.venv\Lib\site-packages\keras\src\layers\layer.py:431: UserWarning: `build()` was called on layer 'vae', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "vae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Train Model

In [8]:
dataset = tf.data.Dataset.from_tensor_slices(x_train)
dataset = dataset.shuffle(BATCH_SIZE * 5).batch(BATCH_SIZE)

In [9]:
kl_div_history = []
loss_history = []

m_batches = x_train.shape[0] 

for epoch in range(MAX_EPOCHS):
    for x in dataset:
        with tf.GradientTape() as tape:
            x = tf.reshape(x, [-1, IMAGE_SIZE]) # flatten image
            with tf.GradientTape() as tape:
                # Reconstruction Loss
                x_reconstruction_logits, mu, log_var = model(x)
                reconstruction_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=x, logits=x_reconstruction_logits)
                reconstruction_loss = tf.reduce_sum(reconstruction_loss) / BATCH_SIZE

                # KL Divergence
                kl_div = -0.5 * tf.reduce_sum(1. + log_var - tf.square(mu) - tf.exp(log_var), axis=-1)
                kl_div = tf.reduce_mean(kl_div)

                # Reconstruction Loss + KL Divergence
                loss = tf.reduce_mean(reconstruction_loss + kl_div)
                gradients = tape.gradient(loss, model.trainable_variables)
                for g in gradients:
                    tf.clip_by_norm(g, 15) # gradient clipping
                
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            
            print(f"Epoch: [{epoch+1} / {MAX_EPOCHS}] Reconstruction Loss: {float (reconstruction_loss):.2f} KL Divergence: {float(kl_div):.2f} Total Loss: {float(loss):.2f}")

            kl_div_history.append(float(kl_div))
            loss_history.append(float(loss))

Epoch: [1 / 10] Reconstruction Loss: 24957.48 KL Divergence: 43.53 Total Loss: 25001.01
Epoch: [1 / 10] Reconstruction Loss: 24866.98 KL Divergence: 646.74 Total Loss: 25513.72
Epoch: [1 / 10] Reconstruction Loss: 24750.06 KL Divergence: 265.33 Total Loss: 25015.38
Epoch: [1 / 10] Reconstruction Loss: 24729.56 KL Divergence: 313.11 Total Loss: 25042.67
Epoch: [1 / 10] Reconstruction Loss: 25051.99 KL Divergence: 300.16 Total Loss: 25352.16
Epoch: [1 / 10] Reconstruction Loss: 24752.32 KL Divergence: 307.61 Total Loss: 25059.93
Epoch: [1 / 10] Reconstruction Loss: 24132.41 KL Divergence: 314.82 Total Loss: 24447.23
Epoch: [1 / 10] Reconstruction Loss: 24194.20 KL Divergence: 311.05 Total Loss: 24505.25
Epoch: [1 / 10] Reconstruction Loss: 24250.23 KL Divergence: 309.48 Total Loss: 24559.72
Epoch: [1 / 10] Reconstruction Loss: 23961.62 KL Divergence: 327.02 Total Loss: 24288.63
Epoch: [1 / 10] Reconstruction Loss: 23825.47 KL Divergence: 346.18 Total Loss: 24171.65
Epoch: [1 / 10] Recons

KeyboardInterrupt: 